# Sri Lankan Tea Factory Reasonable Price Prediction — V2 Cleaned & Safer Forecasting

**Goal:** predict the **next month's `reasonablePrice` for a selected factory** using only information known up to the previous month.

### V2 fixes included
1. Low-volume factory-month price observations are flagged and excluded from price targets/history.
2. Threshold sensitivity report for 50 / 100 / 250 / 500 / 1000 quantity cut-offs.
3. Minimum reliable-history rule for factories.
4. Direct-price, price-delta, and percentage-change targets are compared.
5. Time-aware train/validation/test splitting and rolling-year tuning remain in place.
6. Final 2025 testing remains untouched until validation-based model selection is complete.
7. Prediction function and FastAPI block unreliable previous-month anchors and weak-history factories.
8. V1 outputs are preserved because V2 writes to `colab_outputs_v2_cleaned`.

> Dataset folder: `/content/drive/MyDrive/Tea Prediction`

> For LSTM, use a T4 GPU if available. Tabular models work on CPU.


In [ ]:
# ============================================================
# 0. INSTALL DEPENDENCIES
# ============================================================
# Run this cell once at the beginning of a fresh Colab runtime.

!pip -q install catboost shap fastapi uvicorn nest-asyncio joblib httpx

In [ ]:
# ============================================================
# 0.1 MOUNT GOOGLE DRIVE + GLOBAL CONFIGURATION
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, glob, json, math, random, warnings, itertools, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ---------------- USER SETTINGS ----------------
DATA_DIR = Path('/content/drive/MyDrive/Tea Prediction')
OUTPUT_DIR = DATA_DIR / 'colab_outputs_v2_cleaned'
MODEL_DIR = OUTPUT_DIR / 'models'
RESULT_DIR = OUTPUT_DIR / 'results'
API_DIR = OUTPUT_DIR / 'api'

for p in [OUTPUT_DIR, MODEL_DIR, RESULT_DIR, API_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Set False later if you want a more extensive hyperparameter search.
FAST_MODE = True
RUN_TUNING = True
RUN_SHAP = True
RUN_LSTM = True

# V2 reliability settings
MIN_SALE_QTY = 100
MIN_RELIABLE_MONTHS = 36
MIN_FEATURE_COMPLETENESS = 0.60
SENSITIVITY_THRESHOLDS = [50, 100, 250, 500, 1000]

# Time split based on the MONTH BEING PREDICTED (target year)
TRAIN_END_YEAR = 2022
VALIDATION_START_YEAR = 2023
VALIDATION_END_YEAR = 2024
TEST_YEAR = 2025

print('DATA_DIR   :', DATA_DIR)
print('OUTPUT_DIR :', OUTPUT_DIR)
assert DATA_DIR.exists(), f'Dataset folder does not exist: {DATA_DIR}'

## 1. Dataset discovery and audit

The notebook uses the **Reasonable Price** table as the factory-month base. It reads all uploaded supporting datasets for auditing, but it deliberately avoids merging the same information twice. In particular, `tea_market_features_monthly...csv` is already a consolidated table containing export, international tea-price and prior-year destination indicators.

In [ ]:
# ============================================================
# 1.1 FIND FILES AUTOMATICALLY
# ============================================================
def find_file(patterns, required=True):
    if isinstance(patterns, str):
        patterns = [patterns]
    hits = []
    for pattern in patterns:
        hits.extend(glob.glob(str(DATA_DIR / pattern)))
    hits = sorted(set(hits))
    if not hits:
        if required:
            raise FileNotFoundError(f'Could not find file matching: {patterns} in {DATA_DIR}')
        return None
    if len(hits) > 1:
        print(f'⚠️ Multiple matches for {patterns}; using: {Path(hits[0]).name}')
    return Path(hits[0])

FILES = {
    'reasonable_price': find_file(['*Resonable*Price*2015*2025*.csv', '*Reasonable*Price*2015*2025*.csv']),
    'tc5': find_file('*TC5*2015*2025*.csv'),
    'weather': find_file('*ratnapura*weather*.csv'),
    'market_features': find_file('*tea_market_features_monthly*.csv'),
    'international_demand': find_file('*international_tea_demand_factors_monthly*.csv'),
    'exports_monthly': find_file('*tea_exports_monthly*.csv'),
    'destination_indicators': find_file('*tea_destination_indicators_annual*.csv'),
    'exports_by_destination': find_file('*tea_exports_by_destination_annual*.csv'),
    'usd_lkr': find_file('*usd_lkr*.csv'),
}

for key, path in FILES.items():
    print(f'{key:24s} -> {path.name}')

In [ ]:
# ============================================================
# 1.2 LOAD ALL DATASETS
# ============================================================
data = {name: pd.read_csv(path) for name, path in FILES.items()}

for name, df in data.items():
    print(f'\n{name}: shape={df.shape}')
    print('Columns:', list(df.columns))

In [ ]:
# ============================================================
# 1.3 AUDIT FUNCTIONS
# ============================================================
def safe_date_range(df):
    temp = df.copy()
    if {'Year', 'Month'}.issubset(temp.columns):
        d = pd.to_datetime(dict(year=temp['Year'], month=temp['Month'], day=1), errors='coerce')
    elif {'ProdYear', 'ProdMonth'}.issubset(temp.columns):
        d = pd.to_datetime(dict(year=temp['ProdYear'], month=temp['ProdMonth'], day=1), errors='coerce')
    elif 'date' in temp.columns:
        # Try ISO first, then US-style daily date used in the weather dataset.
        d = pd.to_datetime(temp['date'], errors='coerce')
        if d.isna().mean() > 0.20:
            d = pd.to_datetime(temp['date'], format='%m/%d/%Y', errors='coerce')
    else:
        return None, None
    return d.min(), d.max()

rows=[]
for name, df in data.items():
    dmin, dmax = safe_date_range(df)
    rows.append({
        'dataset': name,
        'rows': len(df),
        'columns': df.shape[1],
        'date_min': dmin,
        'date_max': dmax,
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
        'missing_pct': round(float(df.isna().mean().mean()*100), 2),
    })

audit_summary = pd.DataFrame(rows)
display(audit_summary)
audit_summary.to_csv(RESULT_DIR / '01_dataset_audit_summary.csv', index=False)

# Factory overlap audit
rp = data['reasonable_price']
tc5 = data['tc5']
rp_factories = set(rp['Factory'].dropna().astype(str))
tc5_factories = set(tc5['Factory'].dropna().astype(str))

factory_overlap = pd.DataFrame({
    'metric': ['Reasonable Price factories', 'TC5 factories', 'Common factories', 'Price-only factories', 'TC5-only factories'],
    'count': [len(rp_factories), len(tc5_factories), len(rp_factories & tc5_factories), len(rp_factories - tc5_factories), len(tc5_factories - rp_factories)]
})
display(factory_overlap)
factory_overlap.to_csv(RESULT_DIR / '02_factory_overlap.csv', index=False)

In [ ]:
# ============================================================
# 1.4 DETAILED QUALITY CHECKS
# ============================================================
rp = data['reasonable_price'].copy()
rp['date'] = pd.to_datetime(dict(year=rp['Year'], month=rp['Month'], day=1))

tc5 = data['tc5'].copy()
tc5['date'] = pd.to_datetime(dict(year=tc5['ProdYear'], month=tc5['ProdMonth'], day=1))

print('Reasonable Price duplicate Factory-Month keys:', rp.duplicated(['Factory', 'date']).sum())
print('TC5 duplicate Factory-Month keys            :', tc5.duplicated(['Factory', 'date']).sum())

# Missingness tables
rp_missing = (rp.isna().mean()*100).sort_values(ascending=False).rename('missing_pct').reset_index().rename(columns={'index':'column'})
tc5_missing = (tc5.isna().mean()*100).sort_values(ascending=False).rename('missing_pct').reset_index().rename(columns={'index':'column'})

display(Markdown('### Reasonable Price missingness'))
display(rp_missing)
display(Markdown('### TC5 missingness'))
display(tc5_missing)

# Important leakage/relationship diagnostic
numeric_cols = ['NetAvg', 'GrossAvg', 'reasonablePrice']
print('\nCorrelation among current-month auction averages and reasonablePrice:')
display(rp[numeric_cols].corr())
print('⚠️ Same-month NetAvg/GrossAvg should never be used to predict the SAME month reasonablePrice.')
print('✅ In this notebook they are used only from month t to predict month t+1.')

In [ ]:
# ============================================================
# 1.5 EXTERNAL-DATA CONSOLIDATION + LOW-VOLUME PRICE AUDIT
# ============================================================
market_cols = set(data['market_features'].columns)
exports_cols = set(data['exports_monthly'].columns)
international_cols = set(data['international_demand'].columns)

print('Monthly export columns already found in market_features:')
print(sorted((exports_cols & market_cols) - {'date','year','month'}))
print('\nInternational-demand columns already found in market_features:')
print(sorted((international_cols & market_cols) - {'date','year','month','source_id'}))
print('\nDecision: use tea_market_features_monthly as the consolidated market table.')

# Low-volume sensitivity audit
rp_check = data['reasonable_price'].copy()
for c in ['NetQtySold','GrossQtySold','NetAvg','GrossAvg','reasonablePrice']:
    if c in rp_check.columns:
        rp_check[c] = pd.to_numeric(rp_check[c], errors='coerce')

rows=[]
for threshold in SENSITIVITY_THRESHOLDS:
    low = (rp_check['NetQtySold'] < threshold) | (rp_check['GrossQtySold'] < threshold)
    rows.append({
        'threshold': threshold,
        'flagged_rows': int(low.sum()),
        'flagged_pct': round(float(low.mean()*100), 3),
        'remaining_rows': int((~low).sum()),
        'remaining_factories': int(rp_check.loc[~low,'Factory'].astype(str).nunique()),
        'median_price_flagged': float(rp_check.loc[low,'reasonablePrice'].median()) if low.any() else np.nan,
        'median_price_remaining': float(rp_check.loc[~low,'reasonablePrice'].median()) if (~low).any() else np.nan,
    })
threshold_sensitivity = pd.DataFrame(rows)
display(threshold_sensitivity)
threshold_sensitivity.to_csv(RESULT_DIR / '03_low_volume_threshold_sensitivity.csv', index=False)

high = rp_check['reasonablePrice'] > 500
print('\nRows with reasonablePrice > LKR 500/kg:', int(high.sum()))
if high.any():
    display(rp_check.loc[high, ['Factory','FactoryName','Year','Month','NetQtySold','GrossQtySold','reasonablePrice']]
            .sort_values('reasonablePrice', ascending=False).head(30))


## 2. Master dataset creation

One row is created for **one factory × one calendar month**. Missing calendar months are inserted so that a `shift(1)` really means the previous month rather than merely the previous available record.

In [ ]:
# ============================================================
# 2.1 CLEAN REASONABLE PRICE TABLE + CONTINUOUS MONTHLY PANEL
# ============================================================
rp = data['reasonable_price'].copy()
rp['Factory'] = rp['Factory'].astype(str).str.strip()
rp['FactoryName'] = rp['FactoryName'].astype(str).str.strip()
rp['date'] = pd.to_datetime(dict(year=rp['Year'], month=rp['Month'], day=1))

for c in ['NetQtySold','NetProceeds','NetAvg','GrossQtySold','GrossProceeds','GrossAvg','reasonablePrice']:
    rp[c] = pd.to_numeric(rp[c], errors='coerce')

rp_agg = (
    rp.groupby(['Factory', 'FactoryName', 'date'], as_index=False)
      .agg({
          'NetQtySold':'sum',
          'NetProceeds':'sum',
          'NetAvg':'mean',
          'GrossQtySold':'sum',
          'GrossProceeds':'sum',
          'GrossAvg':'mean',
          'reasonablePrice':'mean',
      })
)

# Preserve raw values for audit before cleaning.
rp_agg['reasonablePrice_raw'] = rp_agg['reasonablePrice']
rp_agg['NetAvg_raw'] = rp_agg['NetAvg']
rp_agg['GrossAvg_raw'] = rp_agg['GrossAvg']

# V2 reliability rule: BOTH net and gross sold quantities must meet the threshold.
rp_agg['low_volume_flag'] = (
    (rp_agg['NetQtySold'] < MIN_SALE_QTY) |
    (rp_agg['GrossQtySold'] < MIN_SALE_QTY)
)
rp_agg['reliable_price_observation'] = (
    (~rp_agg['low_volume_flag']) &
    rp_agg['reasonablePrice'].notna() &
    (rp_agg['reasonablePrice'] > 0)
)

# Keep the calendar month and quantities, but remove unreliable price-derived values.
rp_agg.loc[~rp_agg['reliable_price_observation'], ['reasonablePrice','NetAvg','GrossAvg']] = np.nan

flagged = rp_agg.loc[rp_agg['low_volume_flag'], [
    'Factory','FactoryName','date','NetQtySold','GrossQtySold',
    'reasonablePrice_raw','NetAvg_raw','GrossAvg_raw'
]].copy()
flagged.to_csv(RESULT_DIR / '04_low_volume_rows_flagged.csv', index=False)
print('Reliable price observations:', int(rp_agg['reliable_price_observation'].sum()))
print('Flagged/unreliable observations:', int((~rp_agg['reliable_price_observation']).sum()))
display(flagged.sort_values('reasonablePrice_raw', ascending=False).head(25))

# Build a continuous monthly timeline separately for every factory.
panel_parts = []
for factory, g in rp_agg.groupby('Factory'):
    date_index = pd.date_range(g['date'].min(), g['date'].max(), freq='MS')
    panel_parts.append(pd.DataFrame({'Factory': factory, 'date': date_index}))

panel = pd.concat(panel_parts, ignore_index=True)
factory_name_map = rp_agg.sort_values('date').groupby('Factory')['FactoryName'].last()
panel['FactoryName'] = panel['Factory'].map(factory_name_map)
panel = panel.merge(rp_agg.drop(columns=['FactoryName']), on=['Factory','date'], how='left', validate='one_to_one')
panel['reliable_price_observation'] = panel['reliable_price_observation'].fillna(False).astype(bool)
panel['low_volume_flag'] = panel['low_volume_flag'].fillna(False).astype(bool)

# Factory eligibility is based only on cleaned/reliable price history.
factory_quality = (
    panel.groupby(['Factory','FactoryName'], as_index=False)
    .agg(
        calendar_months=('date','count'),
        reliable_price_months=('reasonablePrice','count'),
        first_month=('date','min'),
        last_month=('date','max'),
    )
)
factory_quality['eligible_factory'] = factory_quality['reliable_price_months'] >= MIN_RELIABLE_MONTHS
eligible_factory_set = set(factory_quality.loc[factory_quality['eligible_factory'], 'Factory'].astype(str))
panel['eligible_factory'] = panel['Factory'].astype(str).isin(eligible_factory_set)

print('Continuous factory-month panel:', panel.shape)
print('Factories:', panel['Factory'].nunique())
print('Eligible factories:', int(factory_quality['eligible_factory'].sum()))
print('Date range:', panel['date'].min(), 'to', panel['date'].max())
display(factory_quality.sort_values('reliable_price_months', ascending=False).head(25))
factory_quality.to_csv(RESULT_DIR / '05_factory_quality_and_eligibility.csv', index=False)


In [ ]:
# ============================================================
# 2.2 AGGREGATE TC5 PRODUCTION TO UNIQUE FACTORY-MONTH LEVEL
# ============================================================
tc5 = data['tc5'].copy()
tc5['Factory'] = tc5['Factory'].astype(str).str.strip()
tc5['date'] = pd.to_datetime(dict(year=tc5['ProdYear'], month=tc5['ProdMonth'], day=1))

production_cols = ['ManOwnLeaf', 'ManBoughtLeaf', 'ManOtherEstate', 'TotalProduction', 'DuringTheMonth']
production_monthly = (
    tc5.groupby(['Factory','date'], as_index=False)[production_cols]
       .sum(min_count=1)
)

assert production_monthly.duplicated(['Factory','date']).sum() == 0
panel = panel.merge(production_monthly, on=['Factory','date'], how='left', validate='one_to_one')

print('After TC5 merge:', panel.shape)
print('Production match rate:', round(panel['TotalProduction'].notna().mean()*100, 2), '%')

In [ ]:
# ============================================================
# 2.3 AGGREGATE DAILY RATNAPURA WEATHER TO MONTHLY LEVEL
# ============================================================
weather = data['weather'].copy()

# The uploaded weather file uses month/day/year style such as 1/2/2015.
weather['date'] = pd.to_datetime(weather['date'], format='%m/%d/%Y', errors='coerce')
weather['month_date'] = weather['date'].dt.to_period('M').dt.to_timestamp()

weather_monthly = (
    weather.groupby('month_date', as_index=False)
    .agg(
        avg_temperature=('temperature','mean'),
        avg_humidity=('humidity','mean'),
        total_rainfall=('rainfall','sum'),
        avg_wind_speed=('wind_speed','mean'),
        avg_solar_radiation=('solar_radiation','mean'),
        avg_pressure=('pressure','mean'),
        total_evapotranspiration=('evapotranspiration','sum'),
        weather_days=('date','count'),
    )
    .rename(columns={'month_date':'date'})
)

panel = panel.merge(weather_monthly, on='date', how='left', validate='many_to_one')
print('Weather monthly rows:', len(weather_monthly))
print('After weather merge:', panel.shape)
print('⚠️ Ratnapura weather is treated as a regional proxy unless every factory is inside the represented weather area.')

In [ ]:
# ============================================================
# 2.4 MERGE CONSOLIDATED MONTHLY MARKET FEATURES
# ============================================================
market = data['market_features'].copy()
market['date'] = pd.to_datetime(market['date']).dt.to_period('M').dt.to_timestamp()

# Use raw/current month-t variables. Existing lag1_* columns are excluded because we will
# generate lags consistently ourselves in Section 3.
MARKET_BASE_COLS = [
    'export_quantity_kg',
    'export_value_usd',
    'export_unit_value_usd_per_kg',
    'reported_fob_usd_per_kg',
    'export_conversion_lkr_per_usd',
    'world_tea_average_usd_per_kg',
    'tea_colombo_usd_per_kg',
    'tea_kolkata_usd_per_kg',
    'tea_mombasa_usd_per_kg',
    'competitor_tea_average_usd_per_kg',
    'colombo_premium_vs_competitors_pct',
    'brent_crude_oil_usd_per_bbl',
    'prior_year_reported_destination_count',
    'prior_year_top_destination_value_share',
    'prior_year_top5_destination_value_share',
    'prior_year_destination_value_hhi',
]
MARKET_BASE_COLS = [c for c in MARKET_BASE_COLS if c in market.columns]
market_monthly = market[['date'] + MARKET_BASE_COLS].drop_duplicates('date')

panel = panel.merge(market_monthly, on='date', how='left', validate='many_to_one')
print('After market merge:', panel.shape)

In [ ]:
# ============================================================
# 2.5 MERGE MONTHLY USD/LKR
# ============================================================
usd = data['usd_lkr'].copy()
usd['date'] = pd.to_datetime(usd['date'], errors='coerce').dt.to_period('M').dt.to_timestamp()
usd_monthly = usd.groupby('date', as_index=False).agg(usd_lkr=('usd_lkr','mean'))

panel = panel.merge(usd_monthly, on='date', how='left', validate='many_to_one')

# Save clean master BEFORE lag/target engineering.
panel = panel.sort_values(['Factory','date']).reset_index(drop=True)
panel.to_csv(OUTPUT_DIR / 'master_factory_month_base_v2.csv', index=False)

print('MASTER BASE SHAPE:', panel.shape)
print('Saved:', OUTPUT_DIR / 'master_factory_month_base_v2.csv')
display(panel.head())

## 3–4. Feature engineering + next-month target

For a row dated **December 2025**, the target month is **January 2026**. The row may use December 2025 observations and older history, but never January 2026 actual values.

Two target formulations are kept:

- `target_next_month_price`: direct next-month price.
- `target_delta`: next-month price minus current-month price. Predicting the **change** can be more robust when the absolute price level shifts over time; the final displayed result is still a price.

In [ ]:
# ============================================================
# 3.1 TARGET CREATION AND FACTORY PRICE FEATURES
# ============================================================
features = panel.copy().sort_values(['Factory','date']).reset_index(drop=True)

# Month being predicted
features['target_date'] = features['date'] + pd.offsets.MonthBegin(1)

# Because panel is calendar-continuous, shift(-1) means exactly next calendar month.
features['target_next_month_price'] = features.groupby('Factory')['reasonablePrice'].shift(-1)
features['target_delta'] = features['target_next_month_price'] - features['reasonablePrice']
features['target_pct_change'] = np.where(
    features['reasonablePrice'] > 0,
    features['target_next_month_price'] / features['reasonablePrice'] - 1.0,
    np.nan
)

# Historical price lags (relative to the current known feature month t)
for lag in [1, 2, 3, 6, 12]:
    features[f'price_lag_{lag}'] = features.groupby('Factory')['reasonablePrice'].shift(lag)

# Rolling values include current month t, which is known when predicting t+1.
for window in [3, 6, 12]:
    features[f'price_roll_mean_{window}'] = (
        features.groupby('Factory')['reasonablePrice']
        .transform(lambda s: s.rolling(window, min_periods=2).mean())
    )
    features[f'price_roll_std_{window}'] = (
        features.groupby('Factory')['reasonablePrice']
        .transform(lambda s: s.rolling(window, min_periods=2).std())
    )

features['price_change_1'] = features['reasonablePrice'] - features['price_lag_1']
features['price_pct_change_1'] = features.groupby('Factory')['reasonablePrice'].pct_change()

In [ ]:
# ============================================================
# 3.2 PRODUCTION LAGS + ROLLING FEATURES
# ============================================================
for lag in [1, 2, 3, 6, 12]:
    features[f'production_lag_{lag}'] = features.groupby('Factory')['TotalProduction'].shift(lag)

for window in [3, 6, 12]:
    features[f'production_roll_mean_{window}'] = (
        features.groupby('Factory')['TotalProduction']
        .transform(lambda s: s.rolling(window, min_periods=2).mean())
    )

features['production_change_1'] = features['TotalProduction'] - features['production_lag_1']

In [ ]:
# ============================================================
# 3.3 MARKET / WEATHER / FX LAGS
# ============================================================
WEATHER_BASE_COLS = [
    'avg_temperature','avg_humidity','total_rainfall','avg_wind_speed',
    'avg_solar_radiation','avg_pressure','total_evapotranspiration'
]
EXTERNAL_BASE_COLS = WEATHER_BASE_COLS + ['usd_lkr'] + MARKET_BASE_COLS
EXTERNAL_BASE_COLS = [c for c in EXTERNAL_BASE_COLS if c in features.columns]

# Build one unique monthly external table first; otherwise group-shifting after factory expansion
# would incorrectly shift among repeated rows from the same month.
monthly_external = (
    features[['date'] + EXTERNAL_BASE_COLS]
    .drop_duplicates('date')
    .sort_values('date')
    .reset_index(drop=True)
)

for col in EXTERNAL_BASE_COLS:
    for lag in [1, 3, 6]:
        monthly_external[f'{col}_lag_{lag}'] = monthly_external[col].shift(lag)

EXTERNAL_LAG_COLS = [c for c in monthly_external.columns if c not in ['date'] + EXTERNAL_BASE_COLS]
features = features.merge(monthly_external[['date'] + EXTERNAL_LAG_COLS], on='date', how='left', validate='many_to_one')

In [ ]:
# ============================================================
# 3.4 TARGET-MONTH SEASONALITY + TREND
# ============================================================
features['target_year'] = features['target_date'].dt.year
features['target_month'] = features['target_date'].dt.month
features['target_month_sin'] = np.sin(2*np.pi*features['target_month']/12)
features['target_month_cos'] = np.cos(2*np.pi*features['target_month']/12)

origin_year = int(features['target_date'].dt.year.min())
features['trend_month_index'] = (
    (features['target_date'].dt.year - origin_year) * 12 + features['target_date'].dt.month
)

features = features.replace([np.inf, -np.inf], np.nan)

In [ ]:
# ============================================================
# 3.5 DEFINE MODEL FEATURE LIST + LEAKAGE GUARD
# ============================================================
CORE_FACTORY_FEATURES = [
    'Factory',
    'reasonablePrice',
    'NetAvg',
    'GrossAvg',
    'NetQtySold',
    'GrossQtySold',
    'TotalProduction',
    'ManOwnLeaf',
    'ManBoughtLeaf',
    'ManOtherEstate',
    'DuringTheMonth',
]

ENGINEERED_FACTORY_FEATURES = [c for c in features.columns if c.startswith('price_') or c.startswith('production_')]
CALENDAR_FEATURES = ['target_month', 'target_month_sin', 'target_month_cos', 'trend_month_index']
FEATURE_COLS = list(dict.fromkeys(CORE_FACTORY_FEATURES + ENGINEERED_FACTORY_FEATURES + EXTERNAL_BASE_COLS + EXTERNAL_LAG_COLS + CALENDAR_FEATURES))
FEATURE_COLS = [c for c in FEATURE_COLS if c in features.columns]

FORBIDDEN = {
    'target_next_month_price', 'target_delta', 'target_pct_change', 'target_date',
    'NetProceeds', 'GrossProceeds', 'reasonablePrice_raw', 'NetAvg_raw', 'GrossAvg_raw'
}
assert not any(c in FORBIDDEN for c in FEATURE_COLS)

model_df = features[
    features['target_next_month_price'].notna() &
    features['reasonablePrice'].notna() &
    features['eligible_factory'].eq(True)
].copy()
model_df['target_pct_change'] = model_df['target_pct_change'].replace([np.inf, -np.inf], np.nan)

print('Total feature columns:', len(FEATURE_COLS))
print('Labeled V2 modeling rows:', len(model_df))
print('Eligible modeling factories:', model_df['Factory'].nunique())
print('Target range:', model_df['target_date'].min(), 'to', model_df['target_date'].max())

features.to_csv(OUTPUT_DIR / 'prediction_feature_store_v2.csv', index=False)
model_df.to_csv(OUTPUT_DIR / 'master_ml_dataset_labeled_v2.csv', index=False)
print('Saved V2 feature store and labeled ML dataset.')


In [ ]:
# ============================================================
# 3.6 FEATURE MISSINGNESS / SANITY REPORT
# ============================================================
feature_missing = (
    features[FEATURE_COLS]
    .isna().mean().mul(100)
    .sort_values(ascending=False)
    .rename('missing_pct')
    .reset_index().rename(columns={'index':'feature'})
)
display(feature_missing.head(30))
feature_missing.to_csv(RESULT_DIR / '03_feature_missingness.csv', index=False)

# Verify that December 2025 can act as the feature month for January 2026 when present.
jan_2026_candidates = features[
    (features['target_date'] == pd.Timestamp('2026-01-01')) &
    features['reasonablePrice'].notna() &
    features['eligible_factory'].eq(True) &
    features['reliable_price_observation'].eq(True)
]
print('Factories with a usable Dec-2025 anchor for Jan-2026 prediction:', jan_2026_candidates['Factory'].nunique())

## 5. Time-aware train / validation / test split

The split is based on the **target month** rather than the feature month:

- Train: target years ≤ 2022
- Validation: 2023–2024
- Final untouched test: 2025

No random shuffling is used.

In [ ]:
# ============================================================
# 5.1 CHRONOLOGICAL SPLIT
# ============================================================
train_df = model_df[model_df['target_year'] <= TRAIN_END_YEAR].copy()
val_df = model_df[
    (model_df['target_year'] >= VALIDATION_START_YEAR) &
    (model_df['target_year'] <= VALIDATION_END_YEAR)
].copy()
test_df = model_df[model_df['target_year'] == TEST_YEAR].copy()

assert train_df['target_date'].max() < val_df['target_date'].min()
assert val_df['target_date'].max() < test_df['target_date'].min()

split_summary = pd.DataFrame({
    'split':['Train','Validation','Test'],
    'rows':[len(train_df),len(val_df),len(test_df)],
    'target_start':[train_df.target_date.min(),val_df.target_date.min(),test_df.target_date.min()],
    'target_end':[train_df.target_date.max(),val_df.target_date.max(),test_df.target_date.max()],
    'factories':[train_df.Factory.nunique(),val_df.Factory.nunique(),test_df.Factory.nunique()],
})
display(split_summary)
split_summary.to_csv(RESULT_DIR / '04_time_split_summary.csv', index=False)

## 6. Evaluation utilities, baseline and Linear Regression

The **naive persistence baseline** predicts:

> next month's price = current month's price

Every advanced model should be compared against this baseline. A complex model is not automatically better.

In [ ]:
# ============================================================
# 6.1 METRICS + MODEL HELPERS
# ============================================================
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import joblib

CAT_COLS = ['Factory']
NUM_COLS = [c for c in FEATURE_COLS if c not in CAT_COLS]
TARGET_COLUMN = {'direct':'target_next_month_price', 'delta':'target_delta', 'pct':'target_pct_change'}

def regression_metrics(y_true, y_pred, model_name='model'):
    y_true = np.asarray(y_true, dtype=float); y_pred = np.asarray(y_pred, dtype=float)
    return {
        'Model': model_name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAPE_pct': mean_absolute_percentage_error(y_true, y_pred) * 100,
        'R2': r2_score(y_true, y_pred),
    }

def make_scaled_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('scaler', StandardScaler())]), NUM_COLS),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
    ])

def make_tree_preprocessor():
    return ColumnTransformer([
        ('num', SimpleImputer(strategy='median', add_indicator=True), NUM_COLS),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
    ])

def make_catboost_frame(df):
    X = df[FEATURE_COLS].copy(); X['Factory'] = X['Factory'].fillna('UNKNOWN').astype(str); return X

def reconstruct_price(raw_pred, base_price, mode):
    raw_pred = np.asarray(raw_pred, dtype=float); base_price = np.asarray(base_price, dtype=float)
    if mode == 'direct': return raw_pred
    if mode == 'delta': return base_price + raw_pred
    if mode == 'pct': return base_price * (1.0 + raw_pred)
    raise ValueError(mode)

def predict_price(model, df, mode, catboost=False):
    raw = model.predict(make_catboost_frame(df) if catboost else df[FEATURE_COLS])
    return reconstruct_price(raw, df['reasonablePrice'], mode)

def evaluate_model(model, df, name, mode, catboost=False):
    pred = predict_price(model, df, mode, catboost=catboost)
    return regression_metrics(df['target_next_month_price'], pred, name), pred


In [ ]:
# ============================================================
# 6.2 NAIVE BASELINE
# ============================================================
baseline_val_pred = val_df['reasonablePrice'].to_numpy(dtype=float)
baseline_val_metrics = regression_metrics(
    val_df['target_next_month_price'], baseline_val_pred, 'Naive_Last_Month_Price'
)
display(pd.DataFrame([baseline_val_metrics]))

In [ ]:
# ============================================================
# 6.3 LINEAR / RIDGE — DIRECT, DELTA, PERCENTAGE CHANGE
# ============================================================
linear_direct = Pipeline([('preprocessor', make_scaled_preprocessor()), ('model', LinearRegression())])
linear_direct.fit(train_df[FEATURE_COLS], train_df['target_next_month_price'])
linear_direct_val_metrics, linear_direct_val_pred = evaluate_model(linear_direct, val_df, 'Linear_Direct', 'direct')

ridge_delta = Pipeline([('preprocessor', make_scaled_preprocessor()), ('model', Ridge(alpha=10.0))])
ridge_delta.fit(train_df[FEATURE_COLS], train_df['target_delta'])
ridge_val_metrics, ridge_val_pred = evaluate_model(ridge_delta, val_df, 'Ridge_Delta', 'delta')

ridge_pct = Pipeline([('preprocessor', make_scaled_preprocessor()), ('model', Ridge(alpha=10.0))])
ridge_pct_train = train_df[train_df['target_pct_change'].notna()].copy()
ridge_pct.fit(ridge_pct_train[FEATURE_COLS], ridge_pct_train['target_pct_change'])
ridge_pct_metrics, ridge_pct_pred = evaluate_model(ridge_pct, val_df, 'Ridge_Pct', 'pct')

display(pd.DataFrame([baseline_val_metrics, linear_direct_val_metrics, ridge_val_metrics, ridge_pct_metrics]).sort_values('MAE'))


## 7. Random Forest, XGBoost and CatBoost

For the tree models, the notebook predicts **monthly price change (`target_delta`)** and reconstructs the next-month price. This can handle non-stationary price levels better than asking tree models to extrapolate absolute price levels beyond what they saw during early training years.

In [ ]:
# ============================================================
# 7.1 DEFAULT RANDOM FOREST — DELTA + PCT
# ============================================================
def make_default_rf():
    return Pipeline([
        ('preprocessor', make_tree_preprocessor()),
        ('model', RandomForestRegressor(n_estimators=300, max_depth=14, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE)),
    ])
rf_delta = make_default_rf(); rf_delta.fit(train_df[FEATURE_COLS], train_df['target_delta'])
rf_val_metrics, rf_val_pred = evaluate_model(rf_delta, val_df, 'RandomForest_Delta', 'delta')
rf_pct = make_default_rf(); rf_pct_train = train_df[train_df['target_pct_change'].notna()].copy(); rf_pct.fit(rf_pct_train[FEATURE_COLS], rf_pct_train['target_pct_change'])
rf_pct_metrics, rf_pct_pred = evaluate_model(rf_pct, val_df, 'RandomForest_Pct', 'pct')
print(rf_val_metrics); print(rf_pct_metrics)


In [ ]:
# ============================================================
# 7.2 DEFAULT XGBOOST — DELTA + PCT
# ============================================================
def make_default_xgb():
    return Pipeline([
        ('preprocessor', make_tree_preprocessor()),
        ('model', XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.03, subsample=0.90, colsample_bytree=0.90, objective='reg:squarederror', eval_metric='mae', n_jobs=-1, random_state=RANDOM_STATE)),
    ])
xgb_delta = make_default_xgb(); xgb_delta.fit(train_df[FEATURE_COLS], train_df['target_delta'])
xgb_val_metrics, xgb_val_pred = evaluate_model(xgb_delta, val_df, 'XGBoost_Delta', 'delta')
xgb_pct = make_default_xgb(); xgb_pct_train = train_df[train_df['target_pct_change'].notna()].copy(); xgb_pct.fit(xgb_pct_train[FEATURE_COLS], xgb_pct_train['target_pct_change'])
xgb_pct_metrics, xgb_pct_pred = evaluate_model(xgb_pct, val_df, 'XGBoost_Pct', 'pct')
print(xgb_val_metrics); print(xgb_pct_metrics)


In [ ]:
# ============================================================
# 7.3 DEFAULT CATBOOST — DELTA + PCT
# ============================================================
def make_default_cat():
    return CatBoostRegressor(iterations=700, depth=7, learning_rate=0.03, loss_function='MAE', random_seed=RANDOM_STATE, verbose=0, allow_writing_files=False)
cat_delta = make_default_cat(); cat_delta.fit(make_catboost_frame(train_df), train_df['target_delta'], cat_features=['Factory'])
cat_val_metrics, cat_val_pred = evaluate_model(cat_delta, val_df, 'CatBoost_Delta', 'delta', catboost=True)
cat_pct = make_default_cat(); cat_pct_train = train_df[train_df['target_pct_change'].notna()].copy(); cat_pct.fit(make_catboost_frame(cat_pct_train), cat_pct_train['target_pct_change'], cat_features=['Factory'])
cat_pct_metrics, cat_pct_pred = evaluate_model(cat_pct, val_df, 'CatBoost_Pct', 'pct', catboost=True)
print(cat_val_metrics); print(cat_pct_metrics)


In [ ]:
# ============================================================
# 7.4 DEFAULT VALIDATION COMPARISON
# ============================================================
default_validation_results = pd.DataFrame([
    baseline_val_metrics, linear_direct_val_metrics, ridge_val_metrics, ridge_pct_metrics,
    rf_val_metrics, rf_pct_metrics, xgb_val_metrics, xgb_pct_metrics, cat_val_metrics, cat_pct_metrics
]).sort_values('MAE').reset_index(drop=True)
display(default_validation_results)
default_validation_results.to_csv(RESULT_DIR / '06_default_validation_model_comparison.csv', index=False)
plt.figure(figsize=(12,5)); plt.bar(default_validation_results['Model'], default_validation_results['MAE']); plt.xticks(rotation=45, ha='right'); plt.ylabel('Validation MAE (LKR/kg)'); plt.title('V2 Default Model Validation Comparison'); plt.tight_layout(); plt.show()


## 8. Time-aware hyperparameter tuning

Tuning uses **only training years (≤2022)**. Rolling/expanding folds validate on later years inside the training period. The 2023–2024 validation set and 2025 final test are not used to select tuning parameters.

In [ ]:
# ============================================================
# 8.1 BUILD EXPANDING-YEAR CV FOLDS INSIDE TRAINING DATA
# ============================================================
from sklearn.model_selection import RandomizedSearchCV, ParameterSampler

# Reset index so CV positional indices are correct.
tune_df = train_df.sort_values('target_date').reset_index(drop=True)

candidate_val_years = [y for y in [2019, 2020, 2021, 2022] if y <= TRAIN_END_YEAR]
rolling_folds = []
for val_year in candidate_val_years:
    tr_idx = np.where(tune_df['target_year'].to_numpy() < val_year)[0]
    va_idx = np.where(tune_df['target_year'].to_numpy() == val_year)[0]
    if len(tr_idx) > 0 and len(va_idx) > 0:
        rolling_folds.append((tr_idx, va_idx))

print('Rolling folds:')
for i, (tr_idx, va_idx) in enumerate(rolling_folds, 1):
    print(
        f'Fold {i}: train target years {tune_df.iloc[tr_idx].target_year.min()}–{tune_df.iloc[tr_idx].target_year.max()} '
        f'({len(tr_idx)} rows) -> validate {tune_df.iloc[va_idx].target_year.unique().tolist()} ({len(va_idx)} rows)'
    )

In [ ]:
# ============================================================
# 8.2 TUNE RANDOM FOREST + XGBOOST
# ============================================================
if RUN_TUNING:
    n_iter = 6 if FAST_MODE else 20

    rf_search_model = Pipeline([
        ('preprocessor', make_tree_preprocessor()),
        ('model', RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE)),
    ])
    rf_param_space = {
        'model__n_estimators': [250, 400, 600, 800],
        'model__max_depth': [6, 10, 14, 18, None],
        'model__min_samples_leaf': [1, 2, 4, 6],
        'model__max_features': ['sqrt', 0.5, 0.8, 1.0],
    }
    rf_search = RandomizedSearchCV(
        rf_search_model,
        rf_param_space,
        n_iter=n_iter,
        scoring='neg_mean_absolute_error',
        cv=rolling_folds,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
    )
    rf_search.fit(tune_df[FEATURE_COLS], tune_df['target_delta'])
    tuned_rf = rf_search.best_estimator_
    print('Best RF params:', rf_search.best_params_)
    print('Best RF rolling-CV delta MAE:', -rf_search.best_score_)

    xgb_search_model = Pipeline([
        ('preprocessor', make_tree_preprocessor()),
        ('model', XGBRegressor(
            objective='reg:squarederror', eval_metric='mae', n_jobs=-1, random_state=RANDOM_STATE
        )),
    ])
    xgb_param_space = {
        'model__n_estimators': [300, 500, 700, 1000],
        'model__max_depth': [2, 3, 4, 5, 6, 8],
        'model__learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08],
        'model__subsample': [0.70, 0.85, 1.0],
        'model__colsample_bytree': [0.60, 0.75, 0.90, 1.0],
        'model__min_child_weight': [1, 3, 5, 8],
        'model__reg_lambda': [1, 3, 5, 10],
    }
    xgb_search = RandomizedSearchCV(
        xgb_search_model,
        xgb_param_space,
        n_iter=n_iter,
        scoring='neg_mean_absolute_error',
        cv=rolling_folds,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
    )
    xgb_search.fit(tune_df[FEATURE_COLS], tune_df['target_delta'])
    tuned_xgb = xgb_search.best_estimator_
    print('Best XGB params:', xgb_search.best_params_)
    print('Best XGB rolling-CV delta MAE:', -xgb_search.best_score_)
else:
    tuned_rf = rf_delta
    tuned_xgb = xgb_delta

In [ ]:
# ============================================================
# 8.3 MANUAL TIME-AWARE CATBOOST TUNING
# ============================================================
# Manual loop avoids version incompatibilities between some CatBoost and scikit-learn releases.

def catboost_cv_mae(params, df, folds):
    fold_maes = []
    for tr_idx, va_idx in folds:
        tr = df.iloc[tr_idx]
        va = df.iloc[va_idx]
        model = CatBoostRegressor(
            **params,
            loss_function='MAE',
            random_seed=RANDOM_STATE,
            verbose=0,
            allow_writing_files=False,
        )
        model.fit(make_catboost_frame(tr), tr['target_delta'], cat_features=['Factory'])
        pred_delta = model.predict(make_catboost_frame(va))
        fold_maes.append(mean_absolute_error(va['target_delta'], pred_delta))
    return float(np.mean(fold_maes))

if RUN_TUNING:
    cat_param_space = {
        'iterations': [300, 500, 700, 900],
        'depth': [4, 5, 6, 7, 8],
        'learning_rate': [0.015, 0.025, 0.04, 0.06],
        'l2_leaf_reg': [1, 3, 5, 8, 12],
        'random_strength': [0.2, 0.5, 1.0, 2.0],
    }
    cat_n_iter = 5 if FAST_MODE else 15
    sampled_cat_params = list(ParameterSampler(cat_param_space, n_iter=cat_n_iter, random_state=RANDOM_STATE))

    cat_tuning_rows = []
    for i, params in enumerate(sampled_cat_params, 1):
        score = catboost_cv_mae(params, tune_df, rolling_folds)
        cat_tuning_rows.append({'trial': i, 'cv_mae': score, **params})
        print(f'CatBoost trial {i}/{len(sampled_cat_params)}: MAE={score:.4f} | {params}')

    cat_tuning_results = pd.DataFrame(cat_tuning_rows).sort_values('cv_mae').reset_index(drop=True)
    display(cat_tuning_results)
    cat_tuning_results.to_csv(RESULT_DIR / '06_catboost_tuning.csv', index=False)
    best_cat_params = {k: cat_tuning_results.iloc[0][k] for k in cat_param_space.keys()}
    best_cat_params['iterations'] = int(best_cat_params['iterations'])
    best_cat_params['depth'] = int(best_cat_params['depth'])

    tuned_cat = CatBoostRegressor(
        **best_cat_params,
        loss_function='MAE',
        random_seed=RANDOM_STATE,
        verbose=0,
        allow_writing_files=False,
    )
    tuned_cat.fit(make_catboost_frame(train_df), train_df['target_delta'], cat_features=['Factory'])
else:
    tuned_cat = cat_delta
    best_cat_params = None

In [ ]:
# ============================================================
# 8.4 VALIDATE TUNED DELTA MODELS + SELECT BEST LEARNED MODEL
# ============================================================
rf_tuned_metrics, rf_tuned_val_pred = evaluate_model(tuned_rf, val_df, 'RandomForest_Tuned_Delta', 'delta')
xgb_tuned_metrics, xgb_tuned_val_pred = evaluate_model(tuned_xgb, val_df, 'XGBoost_Tuned_Delta', 'delta')
cat_tuned_metrics, cat_tuned_val_pred = evaluate_model(tuned_cat, val_df, 'CatBoost_Tuned_Delta', 'delta', catboost=True)

validation_results = pd.DataFrame([
    baseline_val_metrics, linear_direct_val_metrics, ridge_val_metrics, ridge_pct_metrics,
    rf_tuned_metrics, xgb_tuned_metrics, cat_tuned_metrics,
    rf_pct_metrics, xgb_pct_metrics, cat_pct_metrics,
]).sort_values('MAE').reset_index(drop=True)
display(validation_results)
validation_results.to_csv(RESULT_DIR / '07_tuned_validation_model_comparison.csv', index=False)

MODEL_MODE_MAP = {
    'Linear_Direct':'direct', 'Ridge_Delta':'delta', 'Ridge_Pct':'pct',
    'RandomForest_Tuned_Delta':'delta', 'XGBoost_Tuned_Delta':'delta', 'CatBoost_Tuned_Delta':'delta',
    'RandomForest_Pct':'pct', 'XGBoost_Pct':'pct', 'CatBoost_Pct':'pct',
}
ml_candidates = validation_results[validation_results['Model'] != 'Naive_Last_Month_Price'].copy()
BEST_ML_NAME = ml_candidates.iloc[0]['Model']
SELECTED_TARGET_MODE = MODEL_MODE_MAP[BEST_ML_NAME]
BASELINE_WINS = validation_results.iloc[0]['Model'] == 'Naive_Last_Month_Price'
print('Best learned ML model on validation:', BEST_ML_NAME)
print('Selected target mode:', SELECTED_TARGET_MODE)
print('Did naive baseline beat all learned models?', BASELINE_WINS)
if BASELINE_WINS:
    print('⚠️ This is a valid research result. Do not select a model using the 2025 test set.')


## 9. Final 2025 test

The model architecture/hyperparameters are selected from training + validation logic first. Then the selected ML model is retrained on all target months through 2024 and evaluated once on target year 2025.

In [ ]:
# ============================================================
# 9.1 REBUILD SELECTED MODEL AND TRAIN THROUGH 2024
# ============================================================
from sklearn.base import clone
train_val_df = model_df[model_df['target_year'] <= VALIDATION_END_YEAR].copy()
MODEL_REGISTRY = {
    'Linear_Direct': linear_direct,
    'Ridge_Delta': ridge_delta,
    'Ridge_Pct': ridge_pct,
    'RandomForest_Tuned_Delta': tuned_rf,
    'XGBoost_Tuned_Delta': tuned_xgb,
    'CatBoost_Tuned_Delta': tuned_cat,
    'RandomForest_Pct': rf_pct,
    'XGBoost_Pct': xgb_pct,
    'CatBoost_Pct': cat_pct,
}
selected_template = MODEL_REGISTRY[BEST_ML_NAME]
fit_df = train_val_df[train_val_df[TARGET_COLUMN[SELECTED_TARGET_MODE]].notna()].copy()

if 'CatBoost' in BEST_ML_NAME:
    final_test_model = CatBoostRegressor(**selected_template.get_params())
    final_test_model.fit(make_catboost_frame(fit_df), fit_df[TARGET_COLUMN[SELECTED_TARGET_MODE]], cat_features=['Factory'])
else:
    final_test_model = clone(selected_template)
    final_test_model.fit(fit_df[FEATURE_COLS], fit_df[TARGET_COLUMN[SELECTED_TARGET_MODE]])

print('Selected ML model:', BEST_ML_NAME)
print('Target mode:', SELECTED_TARGET_MODE)


In [ ]:
# ============================================================
# 9.2 FINAL TEST METRICS
# ============================================================
def model_price_predict(model, df, model_name, target_mode):
    return predict_price(model, df, target_mode, catboost=('CatBoost' in model_name))

selected_test_pred = model_price_predict(final_test_model, test_df, BEST_ML_NAME, SELECTED_TARGET_MODE)
baseline_test_pred = test_df['reasonablePrice'].to_numpy(dtype=float)
selected_test_metrics = regression_metrics(test_df['target_next_month_price'], selected_test_pred, f'FINAL_{BEST_ML_NAME}')
baseline_test_metrics = regression_metrics(test_df['target_next_month_price'], baseline_test_pred, 'FINAL_Naive_Baseline')
final_test_comparison = pd.DataFrame([selected_test_metrics, baseline_test_metrics]).sort_values('MAE')
display(final_test_comparison)
final_test_comparison.to_csv(RESULT_DIR / '08_final_2025_test_metrics.csv', index=False)


In [ ]:
# ============================================================
# 9.3 FACTORY-WISE TEST ERROR ANALYSIS
# ============================================================
test_predictions = test_df[['Factory','FactoryName','date','target_date','reasonablePrice','target_next_month_price']].copy()
test_predictions['predicted_price'] = selected_test_pred
test_predictions['baseline_price'] = baseline_test_pred
test_predictions['absolute_error'] = np.abs(test_predictions['target_next_month_price'] - test_predictions['predicted_price'])
test_predictions['ape_pct'] = (
    test_predictions['absolute_error'] / test_predictions['target_next_month_price'].replace(0, np.nan) * 100
)

factory_metrics = (
    test_predictions.groupby(['Factory','FactoryName'], as_index=False)
    .agg(
        TestMonths=('target_next_month_price','count'),
        MAE=('absolute_error','mean'),
        MAPE_pct=('ape_pct','mean'),
    )
    .sort_values('MAE')
)

display(factory_metrics.head(15))
display(factory_metrics.tail(15))

test_predictions.to_csv(RESULT_DIR / '09_test_2025_predictions.csv', index=False)
factory_metrics.to_csv(RESULT_DIR / '10_factory_wise_test_metrics.csv', index=False)

In [ ]:
# ============================================================
# 9.4 PREDICTION INTERVAL CALIBRATION FROM VALIDATION RESIDUALS
# ============================================================
# Simple split-conformal style interval using absolute validation residuals.
# Time-series dependence means this is an approximate uncertainty interval, not a guarantee.

# Refit selected architecture on TRAIN only to obtain genuine validation residuals.
if 'CatBoost' in BEST_ML_NAME:
    val_cal_model = CatBoostRegressor(**tuned_cat.get_params())
    fit_df = train_df[train_df[TARGET_COLUMN[SELECTED_TARGET_MODE]].notna()].copy()
    val_cal_model.fit(make_catboost_frame(fit_df), fit_df[TARGET_COLUMN[SELECTED_TARGET_MODE]], cat_features=['Factory'])
else:
    val_cal_model = clone(MODEL_REGISTRY[BEST_ML_NAME])
    fit_df = train_df[train_df[TARGET_COLUMN[SELECTED_TARGET_MODE]].notna()].copy()
    val_cal_model.fit(fit_df[FEATURE_COLS], fit_df[TARGET_COLUMN[SELECTED_TARGET_MODE]])

val_cal_pred = model_price_predict(val_cal_model, val_df, BEST_ML_NAME, SELECTED_TARGET_MODE)
val_abs_residuals = np.abs(val_df['target_next_month_price'].to_numpy(dtype=float) - val_cal_pred)
INTERVAL_ABS_Q90 = float(np.quantile(val_abs_residuals, 0.90))

print(f'Approx. 90% prediction half-width from validation residuals: ± {INTERVAL_ABS_Q90:.2f} LKR/kg')

In [ ]:
# ============================================================
# 9.5 OPTIONAL V1 VS V2 FINAL-TEST COMPARISON
# ============================================================
v1_path = DATA_DIR / 'colab_outputs' / 'results' / '08_final_2025_test_metrics.csv'
if v1_path.exists():
    try:
        v1 = pd.read_csv(v1_path); v1['Version'] = 'V1'
        v2 = final_test_comparison.copy(); v2['Version'] = 'V2_Cleaned'
        comparison = pd.concat([v1, v2], ignore_index=True)
        display(comparison[['Version','Model','MAE','RMSE','MAPE_pct','R2']])
        comparison.to_csv(RESULT_DIR / '11_v1_vs_v2_final_test_comparison.csv', index=False)
    except Exception as e:
        print('V1 comparison could not be created:', e)
else:
    print('V1 final-test CSV not found. V2 is still complete independently.')


## 10. Explainability with SHAP

For delta-based models, SHAP explains **why the model predicts an increase or decrease from the current price**. This is particularly useful for the website's “Why this prediction?” section.

In [ ]:
# ============================================================
# 10.1 SHAP GLOBAL + INDIVIDUAL EXPLANATION
# ============================================================
import shap

SHAP_IMPORTANCE_PATH = RESULT_DIR / '11_shap_feature_importance.csv'

if RUN_SHAP:
    # Use a manageable sample.
    explain_df = test_df.sample(min(300, len(test_df)), random_state=RANDOM_STATE).copy()

    if 'CatBoost' in BEST_ML_NAME:
        from catboost import Pool
        cb_X = make_catboost_frame(explain_df)
        cb_pool = Pool(cb_X, cat_features=['Factory'])
        shap_matrix = final_test_model.get_feature_importance(cb_pool, type='ShapValues')
        shap_values_only = shap_matrix[:, :-1]
        base_values = shap_matrix[:, -1]

        shap_importance = pd.DataFrame({
            'feature': FEATURE_COLS,
            'mean_abs_shap': np.abs(shap_values_only).mean(axis=0),
        }).sort_values('mean_abs_shap', ascending=False)
        display(shap_importance.head(25))
        shap_importance.to_csv(SHAP_IMPORTANCE_PATH, index=False)

        plt.figure(figsize=(9,7))
        top = shap_importance.head(20).sort_values('mean_abs_shap')
        plt.barh(top['feature'], top['mean_abs_shap'])
        plt.xlabel('Mean |SHAP value|')
        plt.title('Global SHAP Importance — CatBoost')
        plt.tight_layout()
        plt.show()

        # One prediction waterfall
        i = 0
        exp = shap.Explanation(
            values=shap_values_only[i],
            base_values=base_values[i],
            data=cb_X.iloc[i].values,
            feature_names=FEATURE_COLS,
        )
        shap.plots.waterfall(exp, max_display=20)

    else:
        pipe = final_test_model
        preprocessor = pipe.named_steps['preprocessor']
        estimator = pipe.named_steps['model']
        transformed = preprocessor.transform(explain_df[FEATURE_COLS])
        feature_names = preprocessor.get_feature_names_out()

        # Convert small sample to dense if needed for SHAP plotting.
        if hasattr(transformed, 'toarray'):
            transformed_for_shap = transformed.toarray()
        else:
            transformed_for_shap = np.asarray(transformed)

        if isinstance(estimator, LinearRegression) or isinstance(estimator, Ridge):
            explainer = shap.LinearExplainer(estimator, transformed_for_shap)
        else:
            explainer = shap.TreeExplainer(estimator)
        sv = explainer(transformed_for_shap)

        shap_importance = pd.DataFrame({
            'feature': feature_names,
            'mean_abs_shap': np.abs(sv.values).mean(axis=0),
        }).sort_values('mean_abs_shap', ascending=False)
        display(shap_importance.head(25))
        shap_importance.to_csv(SHAP_IMPORTANCE_PATH, index=False)

        shap.plots.bar(sv, max_display=20)
        shap.plots.waterfall(sv[0], max_display=20)
else:
    print('RUN_SHAP=False — skipped.')

## 11. LSTM experiment

This is an **experimental sequence model**, not the first model you should trust. It uses the previous 12 months of selected numeric features for each factory and predicts the next-month **price change**, then reconstructs the price.

The same chronological target-year split is retained.

In [ ]:
# ============================================================
# 11.1 PREPARE 12-MONTH FACTORY SEQUENCES
# ============================================================
LSTM_FEATURES = [c for c in [
    'reasonablePrice',
    'NetAvg',
    'GrossAvg',
    'TotalProduction',
    'ManOwnLeaf',
    'ManBoughtLeaf',
    'avg_temperature',
    'avg_humidity',
    'total_rainfall',
    'usd_lkr',
    'export_quantity_kg',
    'export_value_usd',
    'export_unit_value_usd_per_kg',
    'world_tea_average_usd_per_kg',
    'tea_colombo_usd_per_kg',
    'tea_mombasa_usd_per_kg',
    'brent_crude_oil_usd_per_bbl',
] if c in features.columns]

SEQ_LEN = 12

def build_sequences(df, feature_cols, seq_len=12):
    Xs, ys, bases, years, meta = [], [], [], [], []
    for factory, g in df.sort_values(['Factory','date']).groupby('Factory'):
        g = g.reset_index(drop=True)
        for end_idx in range(seq_len-1, len(g)):
            row = g.iloc[end_idx]
            if pd.isna(row['target_next_month_price']) or pd.isna(row['reasonablePrice']):
                continue
            start_idx = end_idx - seq_len + 1
            window = g.loc[start_idx:end_idx, feature_cols].to_numpy(dtype=float)
            # Ensure dates are exactly consecutive months.
            expected = pd.date_range(g.loc[start_idx,'date'], periods=seq_len, freq='MS')
            actual = pd.DatetimeIndex(g.loc[start_idx:end_idx,'date'])
            if not expected.equals(actual):
                continue
            Xs.append(window)
            ys.append(float(row['target_delta']))
            bases.append(float(row['reasonablePrice']))
            years.append(int(row['target_year']))
            meta.append((factory, row['FactoryName'], row['date'], row['target_date']))
    return np.asarray(Xs), np.asarray(ys), np.asarray(bases), np.asarray(years), meta

X_seq, y_seq_delta, seq_base_price, seq_target_year, seq_meta = build_sequences(features, LSTM_FEATURES, SEQ_LEN)
print('Sequences:', X_seq.shape, 'Target:', y_seq_delta.shape)
print('LSTM features:', LSTM_FEATURES)

In [ ]:
# ============================================================
# 11.2 IMPUTE + SCALE USING TRAIN SEQUENCES ONLY
# ============================================================
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

seq_train_mask = seq_target_year <= TRAIN_END_YEAR
seq_val_mask = (seq_target_year >= VALIDATION_START_YEAR) & (seq_target_year <= VALIDATION_END_YEAR)
seq_test_mask = seq_target_year == TEST_YEAR

n_features_lstm = X_seq.shape[2]

lstm_imputer = SimpleImputer(strategy='median')
lstm_scaler = StandardScaler()

train_flat = X_seq[seq_train_mask].reshape(-1, n_features_lstm)
train_flat_imp = lstm_imputer.fit_transform(train_flat)
lstm_scaler.fit(train_flat_imp)


def transform_sequences(X):
    shape = X.shape
    flat = X.reshape(-1, shape[2])
    flat = lstm_imputer.transform(flat)
    flat = lstm_scaler.transform(flat)
    return flat.reshape(shape)

X_seq_scaled = transform_sequences(X_seq)

X_lstm_train = X_seq_scaled[seq_train_mask]
y_lstm_train = y_seq_delta[seq_train_mask]
X_lstm_val = X_seq_scaled[seq_val_mask]
y_lstm_val = y_seq_delta[seq_val_mask]
X_lstm_test = X_seq_scaled[seq_test_mask]
y_lstm_test = y_seq_delta[seq_test_mask]

print('Train:', X_lstm_train.shape, 'Validation:', X_lstm_val.shape, 'Test:', X_lstm_test.shape)

In [ ]:
# ============================================================
# 11.3 TRAIN AND EVALUATE LSTM
# ============================================================
lstm_test_metrics = None

if RUN_LSTM:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

    tf.random.set_seed(RANDOM_STATE)

    lstm_model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, len(LSTM_FEATURES))),
        Dropout(0.20),
        LSTM(32),
        Dropout(0.20),
        Dense(32, activation='relu'),
        Dense(1),
    ])
    lstm_model.compile(optimizer='adam', loss='mae', metrics=['mae'])

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', patience=4, factor=0.5, min_lr=1e-5),
    ]

    history = lstm_model.fit(
        X_lstm_train, y_lstm_train,
        validation_data=(X_lstm_val, y_lstm_val),
        epochs=60 if not FAST_MODE else 35,
        batch_size=64,
        callbacks=callbacks,
        verbose=1,
    )

    test_delta_pred_lstm = lstm_model.predict(X_lstm_test, verbose=0).reshape(-1)
    test_price_pred_lstm = seq_base_price[seq_test_mask] + test_delta_pred_lstm
    test_true_price_lstm = seq_base_price[seq_test_mask] + y_lstm_test
    lstm_test_metrics = regression_metrics(test_true_price_lstm, test_price_pred_lstm, 'LSTM_Delta')
    display(pd.DataFrame([lstm_test_metrics]))

    plt.figure(figsize=(8,4))
    plt.plot(history.history['loss'], label='Train MAE')
    plt.plot(history.history['val_loss'], label='Validation MAE')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.title('LSTM Training History')
    plt.legend()
    plt.tight_layout()
    plt.show()

    lstm_model.save(MODEL_DIR / 'lstm_delta_model.keras')
    joblib.dump(lstm_imputer, MODEL_DIR / 'lstm_imputer.joblib')
    joblib.dump(lstm_scaler, MODEL_DIR / 'lstm_scaler.joblib')
    with open(MODEL_DIR / 'lstm_metadata.json', 'w') as f:
        json.dump({'seq_len':SEQ_LEN, 'features':LSTM_FEATURES, 'target_mode':'delta'}, f, indent=2)
else:
    print('RUN_LSTM=False — skipped.')

## 12. Train and save the deployment model

After the 2025 test has been reported, the selected model is retrained using **all labeled historical targets through 2025**. This deployment model can then use a December 2025 feature row to predict January 2026.

In [ ]:
# ============================================================
# 12.1 TRAIN DEPLOYMENT MODEL ON ALL RELIABLE LABELED HISTORY THROUGH 2025
# ============================================================
deploy_train_df = model_df[model_df['target_year'] <= TEST_YEAR].copy()
selected_template = MODEL_REGISTRY[BEST_ML_NAME]
fit_df = deploy_train_df[deploy_train_df[TARGET_COLUMN[SELECTED_TARGET_MODE]].notna()].copy()

if 'CatBoost' in BEST_ML_NAME:
    deployment_model = CatBoostRegressor(**selected_template.get_params())
    deployment_model.fit(make_catboost_frame(fit_df), fit_df[TARGET_COLUMN[SELECTED_TARGET_MODE]], cat_features=['Factory'])
else:
    deployment_model = clone(selected_template)
    deployment_model.fit(fit_df[FEATURE_COLS], fit_df[TARGET_COLUMN[SELECTED_TARGET_MODE]])

MODEL_PATH = MODEL_DIR / 'reasonable_price_deployment_model_v2.joblib'
joblib.dump(deployment_model, MODEL_PATH)
metadata = {
    'version':'V2_cleaned',
    'model_name': BEST_ML_NAME,
    'target_mode': SELECTED_TARGET_MODE,
    'feature_columns': FEATURE_COLS,
    'categorical_columns': CAT_COLS,
    'price_anchor_column': 'reasonablePrice',
    'trained_target_through_year': TEST_YEAR,
    'interval_abs_q90': INTERVAL_ABS_Q90,
    'min_sale_qty': MIN_SALE_QTY,
    'min_reliable_months': MIN_RELIABLE_MONTHS,
    'min_feature_completeness': MIN_FEATURE_COMPLETENESS,
}
with open(MODEL_DIR / 'model_metadata_v2.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved deployment model:', MODEL_PATH)


In [ ]:
# ============================================================
# 12.2 SAFE ONE-MONTH-AHEAD PREDICTION FUNCTION
# ============================================================
def predict_one_month_ahead(factory, target_year, target_month, return_features=False):
    target_date = pd.Timestamp(year=int(target_year), month=int(target_month), day=1)
    feature_date = target_date - pd.offsets.MonthBegin(1)
    fstr = str(factory)
    row = features[(features['Factory'].astype(str) == fstr) & (features['date'] == feature_date)].copy()

    if row.empty:
        raise ValueError(f'No feature row for factory={factory} at {feature_date.date()}. Only one-month-ahead prediction from an available previous month is supported.')
    if not bool(row['eligible_factory'].iloc[0]):
        raise ValueError(f'Factory {factory} does not have the required {MIN_RELIABLE_MONTHS} reliable historical months.')
    if pd.isna(row['reasonablePrice'].iloc[0]) or not bool(row['reliable_price_observation'].iloc[0]):
        raise ValueError(f'Previous-month Reasonable Price is missing or flagged unreliable for {factory} at {feature_date.date()}. Prediction blocked.')

    history_months = int(features[(features['Factory'].astype(str)==fstr) & (features['date']<=feature_date)]['reasonablePrice'].notna().sum())
    if history_months < MIN_RELIABLE_MONTHS:
        raise ValueError(f'Only {history_months} reliable history months are available; need at least {MIN_RELIABLE_MONTHS}.')

    completeness = float(row[FEATURE_COLS].notna().mean(axis=1).iloc[0])
    if completeness < MIN_FEATURE_COMPLETENESS:
        raise ValueError(f'Feature completeness is {completeness*100:.1f}%, below the {MIN_FEATURE_COMPLETENESS*100:.0f}% minimum.')

    if 'CatBoost' in BEST_ML_NAME:
        raw_pred = float(deployment_model.predict(make_catboost_frame(row))[0])
    else:
        raw_pred = float(deployment_model.predict(row[FEATURE_COLS])[0])
    current_price = float(row['reasonablePrice'].iloc[0])
    predicted_price = float(reconstruct_price([raw_pred], [current_price], SELECTED_TARGET_MODE)[0])
    production_available = bool(row['TotalProduction'].notna().iloc[0]) if 'TotalProduction' in row else False
    warnings_list = []
    if not production_available: warnings_list.append('Current-month TC5 production is missing; model imputation is used.')
    if completeness < 0.80: warnings_list.append('Moderate feature completeness.')
    reliability = 'HIGH' if completeness >= 0.90 and production_available and history_months >= 60 else ('MEDIUM' if completeness >= 0.75 else 'LOW')

    result = {
        'factory': fstr, 'factory_name': str(row['FactoryName'].iloc[0]),
        'feature_month': str(feature_date.date()), 'prediction_month': str(target_date.date()),
        'current_reasonable_price': current_price, 'predicted_reasonable_price': predicted_price,
        'approx_lower_90': max(0.0, predicted_price - INTERVAL_ABS_Q90),
        'approx_upper_90': predicted_price + INTERVAL_ABS_Q90,
        'selected_model': BEST_ML_NAME, 'target_mode': SELECTED_TARGET_MODE,
        'reliability': reliability, 'feature_completeness_pct': round(completeness*100,1),
        'reliable_history_months': history_months, 'production_available': production_available,
        'warnings': warnings_list,
    }
    return (result, row[FEATURE_COLS]) if return_features else result

# MF0083/Rilhena is intentionally used as a safety test. If Dec-2025 is still low-volume, a blocked prediction is correct.
try:
    print(json.dumps(predict_one_month_ahead('MF0083', 2026, 1), indent=2))
except Exception as e:
    print('Safe prediction check:', e)


In [ ]:
# ============================================================
# 12.3 INTERACTIVE FACTORY + MONTH WIDGET FOR COLAB
# ============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output

eligible_pairs = features[features['eligible_factory'].eq(True)][['Factory','FactoryName']].drop_duplicates().sort_values('Factory')
factory_options = [(f'{r.Factory} - {r.FactoryName}', r.Factory) for _, r in eligible_pairs.iterrows()]
factory_widget = widgets.Dropdown(options=factory_options, description='Factory:', layout=widgets.Layout(width='550px'))
year_widget = widgets.IntText(value=2026, description='Year:')
month_widget = widgets.Dropdown(options=[(pd.Timestamp(2000,m,1).strftime('%B'),m) for m in range(1,13)], value=1, description='Month:')
predict_button = widgets.Button(description='Predict', button_style='success')
output_widget = widgets.Output()

def on_predict_clicked(_):
    with output_widget:
        clear_output()
        try:
            r = predict_one_month_ahead(factory_widget.value, year_widget.value, month_widget.value)
            print(f"Factory: {r['factory']} - {r['factory_name']}")
            print(f"Prediction month: {r['prediction_month']}")
            print(f"Current reliable price: LKR {r['current_reasonable_price']:.2f}/kg")
            print(f"Predicted next-month price: LKR {r['predicted_reasonable_price']:.2f}/kg")
            print(f"Approx. 90% interval: LKR {r['approx_lower_90']:.2f} to {r['approx_upper_90']:.2f}/kg")
            print(f"Reliability: {r['reliability']} | Completeness: {r['feature_completeness_pct']}% | Reliable history: {r['reliable_history_months']} months")
            print(f"Model: {r['selected_model']} ({r['target_mode']})")
            for w in r['warnings']: print('⚠️', w)
        except Exception as e:
            print('Prediction blocked/error:', e)

predict_button.on_click(on_predict_clicked)
display(widgets.VBox([factory_widget, widgets.HBox([year_widget, month_widget]), predict_button, output_widget]))


## 13. FastAPI backend generation

The next cells create a small backend that can later be called by a React/Vite frontend.

**Endpoints**
- `GET /health`
- `GET /factories`
- `POST /predict` with `factory`, `year`, and `month`

The generated API loads the saved model, metadata and `prediction_feature_store.csv` from the output folder.

In [ ]:
# ============================================================
# 13.1 GENERATE SAFE FASTAPI app.py
# ============================================================
api_code = r'''from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

BASE_DIR = Path(__file__).resolve().parent
ROOT_DIR = BASE_DIR.parent
MODEL_DIR = ROOT_DIR / "models"
FEATURE_STORE_PATH = ROOT_DIR / "prediction_feature_store_v2.csv"
MODEL_PATH = MODEL_DIR / "reasonable_price_deployment_model_v2.joblib"
METADATA_PATH = MODEL_DIR / "model_metadata_v2.json"

model = joblib.load(MODEL_PATH)
with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)
features = pd.read_csv(FEATURE_STORE_PATH, parse_dates=["date", "target_date"])
FEATURE_COLS = metadata["feature_columns"]
MODEL_NAME = metadata["model_name"]
TARGET_MODE = metadata["target_mode"]
INTERVAL = float(metadata.get("interval_abs_q90", 0.0))
MIN_RELIABLE_MONTHS = int(metadata.get("min_reliable_months", 36))
MIN_FEATURE_COMPLETENESS = float(metadata.get("min_feature_completeness", 0.60))

app = FastAPI(title="Tea Reasonable Price Prediction API — V2", version="2.0.0")

class PredictionRequest(BaseModel):
    factory: str = Field(..., examples=["MF0083"])
    year: int = Field(..., ge=2015, le=2100)
    month: int = Field(..., ge=1, le=12)

def as_bool(v):
    if isinstance(v, bool): return v
    return str(v).strip().lower() in {"true", "1", "yes"}

def reconstruct(raw, base, mode):
    if mode == "direct": return raw
    if mode == "delta": return base + raw
    if mode == "pct": return base * (1.0 + raw)
    raise ValueError(mode)

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME, "version": "V2_cleaned"}

@app.get("/factories")
def factories_list():
    fac = features[features["eligible_factory"].map(as_bool)][["Factory","FactoryName"]].drop_duplicates().sort_values("Factory")
    return fac.to_dict(orient="records")

@app.post("/predict")
def predict(req: PredictionRequest):
    target_date = pd.Timestamp(year=req.year, month=req.month, day=1)
    feature_date = target_date - pd.offsets.MonthBegin(1)
    fstr = str(req.factory)
    row = features[(features["Factory"].astype(str) == fstr) & (features["date"] == feature_date)].copy()
    if row.empty:
        raise HTTPException(status_code=400, detail="No previous-month feature row. Only one-month-ahead prediction from available history is supported.")
    if not as_bool(row["eligible_factory"].iloc[0]):
        raise HTTPException(status_code=400, detail="Factory does not meet minimum reliable-history requirement.")
    if pd.isna(row["reasonablePrice"].iloc[0]) or not as_bool(row["reliable_price_observation"].iloc[0]):
        raise HTTPException(status_code=400, detail="Previous-month Reasonable Price is missing or flagged unreliable; prediction blocked.")

    history = int(features[(features["Factory"].astype(str)==fstr) & (features["date"]<=feature_date)]["reasonablePrice"].notna().sum())
    if history < MIN_RELIABLE_MONTHS:
        raise HTTPException(status_code=400, detail=f"Only {history} reliable history months; need {MIN_RELIABLE_MONTHS}.")
    completeness = float(row[FEATURE_COLS].notna().mean(axis=1).iloc[0])
    if completeness < MIN_FEATURE_COMPLETENESS:
        raise HTTPException(status_code=400, detail=f"Feature completeness {completeness*100:.1f}% is below minimum.")

    X = row[FEATURE_COLS].copy()
    if "CatBoost" in MODEL_NAME:
        X["Factory"] = X["Factory"].fillna("UNKNOWN").astype(str)
    raw = float(model.predict(X)[0])
    current = float(row["reasonablePrice"].iloc[0])
    predicted = float(reconstruct(raw, current, TARGET_MODE))
    production_available = bool(row["TotalProduction"].notna().iloc[0]) if "TotalProduction" in row else False
    reliability = "HIGH" if completeness >= 0.90 and production_available and history >= 60 else ("MEDIUM" if completeness >= 0.75 else "LOW")
    warnings = []
    if not production_available: warnings.append("Current-month TC5 production is missing; model imputation is used.")
    if completeness < 0.80: warnings.append("Moderate feature completeness.")

    return {
        "factory": fstr, "factory_name": str(row["FactoryName"].iloc[0]),
        "feature_month": str(feature_date.date()), "prediction_month": str(target_date.date()),
        "current_reasonable_price": current, "predicted_reasonable_price": predicted,
        "approx_lower_90": max(0.0, predicted-INTERVAL), "approx_upper_90": predicted+INTERVAL,
        "model": MODEL_NAME, "target_mode": TARGET_MODE, "reliability": reliability,
        "feature_completeness_pct": round(completeness*100,1), "reliable_history_months": history,
        "production_available": production_available, "warnings": warnings,
    }
'''

(API_DIR / 'app.py').write_text(api_code)
requirements = '''fastapi
uvicorn
pandas
numpy
scikit-learn
xgboost
catboost
joblib
'''
(API_DIR / 'requirements.txt').write_text(requirements)
print('Created:', API_DIR / 'app.py')
print('Created:', API_DIR / 'requirements.txt')


In [ ]:
# ============================================================
# 13.2 TEST FASTAPI WITHOUT OPENING A PUBLIC PORT
# ============================================================
# This test imports the generated app and calls it through FastAPI TestClient.
# If the runtime reports a TestClient dependency issue, run: !pip -q install httpx

import sys
sys.path.insert(0, str(API_DIR))

try:
    from fastapi.testclient import TestClient
    import importlib
    api_module = importlib.import_module('app')
    client = TestClient(api_module.app)

    print(client.get('/health').json())
    response = client.post('/predict', json={'factory':'MF0083','year':2026,'month':1})
    print('Status:', response.status_code)
    print(response.json())
    print('Note: HTTP 400 can be CORRECT if MF0083 Dec-2025 is flagged unreliable by V2.')
except Exception as e:
    print('API test skipped/failed:', e)
    print('The generated app.py is still available in:', API_DIR)

In [ ]:
# ============================================================
# 13.3 OPTIONAL: START UVICORN INSIDE COLAB
# ============================================================
# This starts a local Colab server. A React app outside the Colab runtime cannot reach it
# unless you expose the port with a tunnel or deploy the API to a real hosting service.
# Stop the cell to terminate the server.

# %cd /content/drive/MyDrive/Tea Prediction/colab_outputs/api
# !uvicorn app:app --host 0.0.0.0 --port 8000

# Final outputs

After a successful run, check:

- `colab_outputs_v2_cleaned/results/03_low_volume_threshold_sensitivity.csv`
- `colab_outputs_v2_cleaned/results/05_factory_quality_and_eligibility.csv`
- `colab_outputs_v2_cleaned/results/07_tuned_validation_model_comparison.csv`
- `colab_outputs_v2_cleaned/results/08_final_2025_test_metrics.csv`
- `colab_outputs_v2_cleaned/results/10_factory_wise_test_metrics.csv`
- `colab_outputs_v2_cleaned/models/reasonable_price_deployment_model_v2.joblib`
- `colab_outputs_v2_cleaned/api/app.py`

### Important
If the cleaned ML model still does not beat the naive baseline, **do not force the result**. Report it honestly as a research finding. The V2 notebook is designed to make that comparison scientifically defensible.


# Proposal Scope Completion — Baselines, Residuals and Interval Coverage

This final research section strengthens the proposal evaluation scope without retraining the deployment model. It adds:

- 12-month seasonal-naive baseline
- residual diagnostics
- month-wise errors
- top prediction errors
- held-out validation of the approximate 90% prediction interval

The final ML target remains **one-month-ahead factory Reasonable Price (LKR/kg)**. Tea-owner **gross earnings** can then be estimated as `predicted price × expected leaf kilograms`.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

V2 = Path("/content/drive/MyDrive/Tea Prediction/colab_outputs_v2_cleaned")
R = V2 / "results"
TEST_FILE = R / "09_test_2025_predictions.csv"
MASTER_FILE = V2 / "master_factory_month_base_v2.csv"
METADATA_FILE = V2 / "models/model_metadata_v2.json"

test_ext = pd.read_csv(TEST_FILE, parse_dates=["date", "target_date"])
master_ext = pd.read_csv(MASTER_FILE, parse_dates=["date"])
metadata_ext = json.loads(METADATA_FILE.read_text())
INTERVAL_Q90 = float(metadata_ext["interval_abs_q90"])
print("Loaded", len(test_ext), "held-out 2025 predictions")

In [ ]:
# 12-month seasonal-naive baseline
lookup = master_ext[["Factory", "date", "reasonablePrice", "reliable_price_observation"]].copy()
lookup["target_date"] = lookup["date"] + pd.DateOffset(years=1)
lookup = lookup.rename(columns={"reasonablePrice":"seasonal_naive_price", "reliable_price_observation":"seasonal_source_reliable"})[["Factory","target_date","seasonal_naive_price","seasonal_source_reliable"]]
seasonal = test_ext.merge(lookup, on=["Factory","target_date"], how="left")
reliable = seasonal["seasonal_source_reliable"].astype(str).str.lower().isin(["true","1","yes"])
seasonal = seasonal[seasonal["seasonal_naive_price"].notna() & reliable].copy()

def metric_row(name, pred):
    y = seasonal["target_next_month_price"].to_numpy(float)
    p = np.asarray(pred, float)
    return {
        "Model": name, "N": len(y),
        "MAE": mean_absolute_error(y,p),
        "RMSE": mean_squared_error(y,p)**0.5,
        "MAPE_pct": np.mean(np.abs((y-p)/y))*100,
        "sMAPE_pct": np.mean(2*np.abs(y-p)/(np.abs(y)+np.abs(p)))*100,
        "R2": r2_score(y,p),
    }

seasonal_comparison = pd.DataFrame([
    metric_row("CatBoost_Pct", seasonal["predicted_price"]),
    metric_row("Naive_Last_Month", seasonal["baseline_price"]),
    metric_row("Seasonal_Naive_12M", seasonal["seasonal_naive_price"]),
])
display(seasonal_comparison)
seasonal_comparison.to_csv(R / "12_seasonal_naive_comparison_2025.csv", index=False)

In [ ]:
# Residual diagnostics and monthly errors
test_ext["residual"] = test_ext["target_next_month_price"] - test_ext["predicted_price"]
test_ext["squared_error"] = test_ext["residual"]**2
test_ext["target_month_num"] = test_ext["target_date"].dt.month

residual_summary = pd.DataFrame([{
    "N": len(test_ext),
    "MeanResidual": test_ext.residual.mean(),
    "MedianResidual": test_ext.residual.median(),
    "ResidualStd": test_ext.residual.std(),
    "MAE": test_ext.absolute_error.mean(),
    "RMSE": np.sqrt(test_ext.squared_error.mean()),
}])
monthly_error = test_ext.groupby("target_month_num").agg(
    N=("absolute_error","size"), MAE=("absolute_error","mean"),
    MedianAE=("absolute_error","median"),
    RMSE=("squared_error", lambda x: np.sqrt(x.mean())), MaxAE=("absolute_error","max")
).reset_index()

display(residual_summary)
display(monthly_error)
residual_summary.to_csv(R / "14_residual_summary_2025.csv", index=False)
monthly_error.to_csv(R / "15_monthly_error_analysis_2025.csv", index=False)
test_ext.nlargest(20, "absolute_error").to_csv(R / "16_top20_largest_errors_2025.csv", index=False)

plt.figure(figsize=(8,5)); plt.hist(test_ext.residual, bins=30); plt.axvline(0); plt.title("2025 Test Residual Distribution"); plt.xlabel("Actual - Predicted (LKR/kg)"); plt.ylabel("Frequency"); plt.show()
plt.figure(figsize=(8,5)); plt.scatter(test_ext.predicted_price, test_ext.residual, s=12, alpha=.6); plt.axhline(0); plt.title("Residuals vs Predicted Price — 2025"); plt.xlabel("Predicted LKR/kg"); plt.ylabel("Residual LKR/kg"); plt.show()

In [ ]:
# Validate the approximate 90% interval on the untouched 2025 test set
test_ext["lower_90"] = np.maximum(0, test_ext["predicted_price"] - INTERVAL_Q90)
test_ext["upper_90"] = test_ext["predicted_price"] + INTERVAL_Q90
test_ext["inside_90"] = (test_ext["target_next_month_price"] >= test_ext["lower_90"]) & (test_ext["target_next_month_price"] <= test_ext["upper_90"])
coverage = test_ext["inside_90"].mean()*100
coverage_table = pd.DataFrame([{
    "NominalInterval_pct": 90.0,
    "HalfWidth_LKR_per_kg": INTERVAL_Q90,
    "TestObservations": len(test_ext),
    "CoveredObservations": int(test_ext.inside_90.sum()),
    "HeldOutCoverage_pct": coverage,
}])
display(coverage_table)
coverage_table.to_csv(R / "17_prediction_interval_coverage_2025.csv", index=False)
print(f"Held-out 2025 interval coverage: {coverage:.2f}%")

## Interpretation for the dissertation

- Report the seasonal-naive comparison on the **same subset** where a reliable 12-month prior price exists.
- Residuals should be discussed for bias, dispersion, seasonal difficult periods, and large-error cases.
- The approximate 90% interval is empirical and validation-residual based. Its **held-out test coverage** must be reported rather than assumed.
- A coverage much higher than 90% means the interval is conservative/wide, not that the point forecast is 97% accurate.
- SHAP explanations describe model contribution/association and do not establish causal effects.